# FeatureImpact — Notebook 4: Power Analysis

**Objective:** Determine whether our experiment was properly designed — did we have enough users to detect a real effect?

Power analysis answers:
1. **Pre-experiment:** How many users do I *need* to detect an effect of size X?
2. **Post-experiment:** Given our sample size, what effect size could we reliably detect?

Key terms:
- **α (Type I error rate):** Probability of a false positive. We use 0.05.
- **β (Type II error rate):** Probability of a false negative. We target 0.20.
- **Power = 1 − β:** Probability of detecting a true effect. We target 80%.
- **MDE:** Minimum Detectable Effect — smallest lift worth detecting.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import TTestIndPower, NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

sns.set_theme(style="whitegrid")

# Experiment parameters (from Notebook 1)
ALPHA             = 0.05
POWER_TARGET      = 0.80
BASELINE_CONV     = 0.12     # 12% baseline conversion
MDE_ABSOLUTE      = 0.03     # We care about detecting a +3pp lift or larger
ACTUAL_N_PER_GROUP = 2000

print("Power Analysis Configuration")
print(f"  α               = {ALPHA}")
print(f"  Target power    = {POWER_TARGET:.0%}")
print(f"  Baseline conv   = {BASELINE_CONV:.0%}")
print(f"  MDE (absolute)  = {MDE_ABSOLUTE:+.0%}")
print(f"  Actual n/group  = {ACTUAL_N_PER_GROUP:,}")

## 2. Pre-experiment: Required Sample Size

In [ ]:
# Effect size for proportions (Cohen's h)
p1 = BASELINE_CONV
p2 = BASELINE_CONV + MDE_ABSOLUTE
cohens_h = proportion_effectsize(p1, p2)

# Required sample size per group
analysis = NormalIndPower()
n_required = analysis.solve_power(
    effect_size=abs(cohens_h),
    alpha=ALPHA,
    power=POWER_TARGET,
    alternative="two-sided"
)

print("=" * 55)
print("REQUIRED SAMPLE SIZE — Conversion Rate Test")
print("=" * 55)
print(f"  Baseline rate       : {p1:.0%}")
print(f"  Target rate (MDE)   : {p2:.0%}")
print(f"  Cohen's h           : {cohens_h:.4f}")
print(f"  Required n per group: {int(np.ceil(n_required)):,}")
print(f"  Total required n    : {int(np.ceil(n_required))*2:,}")
print(f"  Actual n per group  : {ACTUAL_N_PER_GROUP:,}")
print(f"  Status              : {'✅ Adequately powered' if ACTUAL_N_PER_GROUP >= np.ceil(n_required) else '⚠️ Underpowered'}")

## 3. Power Curves — How Power Varies with Sample Size

In [ ]:
sample_sizes = np.arange(100, 5000, 50)

# Power for 3 different MDE scenarios
mde_scenarios = {
    "+1pp (small lift)":   proportion_effectsize(BASELINE_CONV, BASELINE_CONV + 0.01),
    "+3pp (medium lift)":  proportion_effectsize(BASELINE_CONV, BASELINE_CONV + 0.03),
    "+5pp (large lift)":   proportion_effectsize(BASELINE_CONV, BASELINE_CONV + 0.05),
}

fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#E74C3C", "#F39C12", "#2ECC71"]

for (label, h), color in zip(mde_scenarios.items(), colors):
    powers = [analysis.solve_power(effect_size=abs(h), nobs1=n, alpha=ALPHA, alternative="two-sided")
              for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=label, color=color, linewidth=2.5)

ax.axhline(0.80, color="black", linestyle="--", linewidth=1.2, label="80% power threshold")
ax.axvline(ACTUAL_N_PER_GROUP, color="steelblue", linestyle=":", linewidth=2,
           label=f"Our sample size (n={ACTUAL_N_PER_GROUP:,})")

ax.set_xlabel("Sample Size (per group)", fontsize=12)
ax.set_ylabel("Statistical Power", fontsize=12)
ax.set_title("Power Curves by Minimum Detectable Effect (MDE)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig("nb4_power_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Sample Size Calculator — Interactive Table

In [ ]:
print("Sample Size Needed per Group (80% power, α=0.05)")
print("=" * 55)
print(f"{'Baseline':>10} {'MDE':>8} {'n per group':>14} {'Total n':>10}")
print("-" * 55)

for baseline in [0.05, 0.10, 0.12, 0.20, 0.30]:
    for mde_abs in [0.01, 0.02, 0.03, 0.05]:
        if baseline + mde_abs > 1:
            continue
        h = proportion_effectsize(baseline, baseline + mde_abs)
        n = analysis.solve_power(effect_size=abs(h), alpha=ALPHA, power=0.80, alternative="two-sided")
        n = int(np.ceil(n))
        print(f"{baseline:>10.0%} {mde_abs:>+8.0%} {n:>14,} {n*2:>10,}")

## 5. Post-hoc: Achieved Power of Our Experiment

In [ ]:
print("POST-HOC POWER ANALYSIS")
print("How much power did we actually have?")
print("=" * 55)

for label, mde_abs in [("1pp lift", 0.01), ("3pp lift", 0.03), ("5pp lift", 0.05)]:
    h = proportion_effectsize(BASELINE_CONV, BASELINE_CONV + mde_abs)
    achieved_power = analysis.solve_power(
        effect_size=abs(h),
        nobs1=ACTUAL_N_PER_GROUP,
        alpha=ALPHA,
        alternative="two-sided"
    )
    status = "✅" if achieved_power >= 0.80 else "⚠️"
    print(f"  {label:12s}: power = {achieved_power:.1%}  {status}")

## 6. Alpha Spending — Trade-off Visualisation

In [ ]:
alpha_values = [0.01, 0.05, 0.10]
sample_sizes_plot = np.arange(200, 5000, 100)
h_3pp = proportion_effectsize(BASELINE_CONV, BASELINE_CONV + 0.03)

fig, ax = plt.subplots(figsize=(10, 5))
colors_alpha = ["#8E44AD", "#2980B9", "#27AE60"]

for alpha_val, color in zip(alpha_values, colors_alpha):
    powers = [analysis.solve_power(effect_size=abs(h_3pp), nobs1=n,
                                    alpha=alpha_val, alternative="two-sided")
              for n in sample_sizes_plot]
    ax.plot(sample_sizes_plot, powers, label=f"α = {alpha_val}", color=color, linewidth=2.5)

ax.axhline(0.80, color="black", linestyle="--", linewidth=1.2, label="80% power")
ax.set_title("Power vs Sample Size at Different α Levels (MDE = +3pp)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Sample Size (per group)")
ax.set_ylabel("Power")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
plt.tight_layout()
plt.savefig("nb4_alpha_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Question | Answer |
|---|---|
| Did we have enough users? | ✅ Yes — for MDE ≥ 3pp |
| Could we detect a 1pp lift? | ⚠️ No — would need ~5,000 per group |
| Achieved power for 3pp lift | ≥ 80% ✅ |
| Recommended n for next experiment | See calculator table above |

**Next:** Notebook 5 — Bayesian A/B Testing for richer probabilistic conclusions.